# **Entreno de ángulos mediante el algoritmo de Ben Priestley**

En la siguiente sección voy a realizar un preentreno de ángulos para instancias pequeñas de CVP basados en la factorización de N

In [1]:
%load_ext autoreload
%autoreload 2

import time
import sympy
import numpy as np
from math import log, log2

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pickle

from Crypto.Util import number


from tqdm.notebook import tqdm

import random

from scipy.optimize import curve_fit, least_squares

#Importar de función interna
from modules import schnorr_lattice as sl
from modules import qaoa
from modules import utils

from modules.functions import solve_cvp, solve_cvp_with_opt_paramters

from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings('ignore')

In [2]:
seed = 42
random.seed = seed

In [3]:
def generate_N(bitLength):
    """
    param bitLength: cantidad de bits de N

    return: N = p*q de bitLength bits
    """
    
    min_p_bit_length = max(2, bitLength // 4)
    max_p_bit_length = bitLength - min_p_bit_length

    

    while True:
        p_bit_length = random.randint(min_p_bit_length, max_p_bit_length)

        p = number.getPrime(p_bit_length)

        q_bit_length = bitLength - p_bit_length + 1

        q = number.getPrime(q_bit_length)

        N = p * q

        if N.bit_length() == bitLength:
            return N


## **Generación de set de entrenamiento**

Código de generación del set de entrenamiento

In [4]:
def generate_train_set(min, max, set_length):
    train_set = []
    bitLengths = []
    
    for _ in range(set_length):
        bitlength = random.randint(min, max)
        N = generate_N(bitlength)
        train_set.append(N)
        bitLengths.append(bitlength)

    
    train_df = pd.DataFrame({
        'N': train_set,
        'Bit_length': bitLengths
    })

    return train_df


## **Generación de set de validación**

Código de generación del set de validación

In [5]:
def generate_val_set(min, max, n_per_length):
    validation_set = []


    for length in range(min, max):
        for _ in range(n_per_length):
            N = generate_N(length)
            validation_set.append(N)

    val_df = pd.DataFrame({
        'N': validation_set,
        'bitLength': [n.bit_length() for n in validation_set]
    })

    return val_df



## **Proceso de entrenamiento**

In [6]:
#Función para ver el escalado de la probabilidad de obtener una mejor solución del paper de Ben Priestley
def scaling_function(x, alpha):
    return 1/(2**(alpha*x))

In [7]:
def normalize_angles(angles: list[list[float]]):
    #Normalizo los angulos en el intervalo: [-π, π]
    nangles = np.array(angles)
    nangles = (nangles + np.pi) % (2*np.pi) - np.pi
    return nangles


def obtain_kmeans(k: int, angles: np.ndarray) -> list[list[float]]:
    kmeans = KMeans(n_clusters = k, random_state = seed, n_init="auto")

    n_angles = angles.shape[1]
    features = np.hstack([
        np.column_stack([np.cos(angles[:, i]), np.sin(angles[:, i])])
        for i in range(n_angles)
    ])

    kmeans.fit(features)
    c = kmeans.cluster_centers_
    centroides = np.column_stack([
        np.arctan2(c[:, 2*i + 1], c[:, 2*i])
        for i in range(n_angles)
    ])

    return centroides.tolist()

In [8]:
def qaoa_pretrain_algorithm(train_set: np.ndarray, val_set: np.ndarray, epochs: int = 5, 
                            clusterize: bool = True, clusters: int = 10,
                            delta: float = 0.75, l: int = 1, c: int = 3, min_method: str = 'Nelder-Mead', p: int = 1,
                            normalizeHc: bool = False, shots: int = 10_000, q: int = 10,
                            seed: int = 42, verbose: bool = False, progress_bar: bool = False):
    """
    TODO
    """
    # Inicializacion
    np.random.seed = seed
    a_op = None #Asignación de angulos inicial
    best_alpha = np.inf
    val_stats = {}

    #train_len = train_set.size
    #val_len = val_set.size

    
    for epoch_idx in tqdm(range(1, epochs + 1), desc = "Epochs") if progress_bar else range(1, epochs + 1): #Episodios de entreno

        tic = time.perf_counter()


        angles_population = [] #Conjunto con los angulos calculados para cada instancia de entreno
        angles_clusters_rep = [] #Conjunto con los representantes de los clusters 

        #Entrenamiento
        for tN in tqdm(train_set, desc = "Training") if progress_bar else train_set: 
            
            cvp = sl.schnorrCVP(tN, c, l, seed, set_seed = False, verbose = False)#Genero una instancia para realizar los calculos
            instance = cvp.generate_cvp(q, verbose = False)

            if a_op is not None: x0 = np.asarray(a_op)
            else: x0 = np.asarray([0.0]*(2*p))

            #Obtengo los angulos optimos
            _,_,_, angles = solve_cvp(cvp, instance, x0, delta, shots = 1, normalize = normalizeHc, p = p, min_method= min_method)

            angles_population.append(list(angles.values()))
        #Fin entrenamiento
        
        #Division de Clusters y sus representantes
        #TODO
        if clusterize:
            normalized_angles = normalize_angles(angles_population)
            angles_clusters_rep = obtain_kmeans(clusters, normalized_angles)
        else:
            angles_clusters_rep = angles_population.copy()
        #---------------------------------------


        #Validacion
        alphas = []
        for a in tqdm(angles_clusters_rep, desc = "Optimal Angles") if progress_bar else angles_clusters_rep: #Por cada angulo representante de su cluster
            lattice_dimension = []
            probabilities = []


            for vN in tqdm(val_set, desc = "Validation") if progress_bar else val_set: #Por cada instancia de validacion
                
                cvp = sl.schnorrCVP(vN, c, l, seed, set_seed = False, verbose = False)
                instance = cvp.generate_cvp(q, verbose = False)

                vnews, probs, _ , _ = solve_cvp_with_opt_paramters(cvp, instance, a, delta, shots, normalize = normalizeHc, p = p)
                dists2 = utils.get_distances2(vnews, instance.t)
                
                best_dist = np.inf
                best_prob = 0.0

                for d, prob in zip(dists2, probs):
                    if d < best_dist:
                        best_dist = d
                        best_prob = prob

                lattice_dimension.append(cvp.n)
                probabilities.append(best_prob)
                
                
            alpha = curve_fit(scaling_function, xdata = lattice_dimension, ydata = probabilities)[0][0]
            alphas.append(alpha)
        #Fin de validacion

        tac = time.perf_counter()

        # Epoch statistics.
        min_alpha = min(alphas)
        mean_alpha = np.mean(alphas)
        std_alpha = np.std(alphas)
        max_alpha = max(alphas)
        runtime = tac - tic
        epoch_stats = [min_alpha, mean_alpha, std_alpha, max_alpha, runtime]
        val_stats[epoch_idx] = epoch_stats

        if verbose:
            print(f'Epoch {epoch_idx:02d} ({int(runtime):04d} s): min - {min_alpha:.3f}, mean - {mean_alpha:.3f}, std - {std_alpha:.3f}, max - {max_alpha:.3f}')

        if min_alpha < best_alpha:
            best_alpha = min_alpha
            a_op = angles_clusters_rep[np.argmin(alphas)]

    return a_op, best_alpha, val_stats
    

### **Entrenamiento 1**

In [9]:
train_set_length = 10
min_train_bitLength = 8
max_train_bitLength = 128

min_val_bitLength = 8
max_val_bitLength = 128
n_per_length = 2
val_set_length = (128 - 8 + 1)*2


In [10]:
train_df = generate_train_set(min_train_bitLength, max_train_bitLength, train_set_length)

train_df.to_csv(f'./training_sets/training_set_size{train_set_length}_randomBitLength_{min_train_bitLength}_to_{max_train_bitLength}.csv', index = False)

In [11]:
val_df = generate_val_set(min_val_bitLength, max_val_bitLength + 1, n_per_length)

val_df.to_csv(f'./validation_sets/validation_set_size{val_set_length}_bitLength_{min_val_bitLength}_to_{max_val_bitLength}.csv', index = False)

In [12]:
train_df = pd.read_csv(f'./training_sets/training_set_size{train_set_length}_randomBitLength_{min_train_bitLength}_to_{max_train_bitLength}.csv')
val_df = pd.read_csv(f'./validation_sets/validation_set_size{val_set_length}_bitLength_{min_val_bitLength}_to_{max_val_bitLength}.csv')

In [13]:
train_set = train_df["N"].to_numpy()
val_set = val_df["N"].to_numpy()

#### Nelder-Mead

In [14]:
val_stats_list = []
optimal_assignments = []
best_alphas = {}
for p in range(1, 11):
    print(f"Para el numero de capas p = {p}")
    
    optimal_params, best_alpha, val_stats = qaoa_pretrain_algorithm(train_set, val_set, epochs= 5, clusterize = False, clusters = 10,
                                                                     delta = 0.75, l = 1, c = 3, min_method = 'Nelder-Mead',
                                                                     p = p, normalizeHc = True,
                                                                    shots = 10_000, seed = 42, verbose = True)
    
    for epoch_idx, epoch_stats in val_stats.items():
        val_stats_list.append([p, epoch_idx] + epoch_stats)
    optimal_assignments.append(optimal_params)
    best_alphas[f"{p}"] = best_alpha
    print(f"Los parametros mas optimos proporcionan un escalado de {best_alpha}")


Para el numero de capas p = 1
Epoch 01 (0181 s): min - 0.432, mean - 0.475, std - 0.029, max - 0.517
Epoch 02 (0171 s): min - 0.451, mean - 0.487, std - 0.049, max - 0.620
Epoch 03 (0180 s): min - 0.445, mean - 0.501, std - 0.046, max - 0.587
Epoch 04 (0172 s): min - 0.434, mean - 0.467, std - 0.027, max - 0.530
Epoch 05 (0171 s): min - 0.435, mean - 0.465, std - 0.034, max - 0.542
Los parametros mas optimos proporcionan un escalado de 0.43171427698789716
Para el numero de capas p = 2
Epoch 01 (0315 s): min - 0.321, mean - 0.377, std - 0.061, max - 0.502
Epoch 02 (0221 s): min - 0.306, mean - 0.410, std - 0.148, max - 0.759
Epoch 03 (0210 s): min - 0.293, mean - 0.335, std - 0.023, max - 0.367
Epoch 04 (0213 s): min - 0.312, mean - 0.351, std - 0.036, max - 0.434
Epoch 05 (0210 s): min - 0.317, mean - 0.333, std - 0.031, max - 0.423
Los parametros mas optimos proporcionan un escalado de 0.293251305857349
Para el numero de capas p = 3
Epoch 01 (0484 s): min - 0.314, mean - 0.515, std - 

In [15]:
val_stats_df = pd.DataFrame(val_stats_list, columns = ['p', 'epoch', 'min_alpha', 'mean_alpha', 'std_alpha', 'max_alpha', 'runtime'])
val_stats_df.to_csv(f"./results/validation_statistics/train{train_set_length}_val{val_set_length}_1_NelderMead.csv", index = False)
best_alphas_df = pd.DataFrame.from_dict(best_alphas, orient = "index", columns=["best_alphas"])
best_alphas_df.index.name = "p"
best_alphas_df.to_csv(f"./results/training_best_alphas/train{train_set_length}_val{val_set_length}_1_NelderMead.csv")

for i, opt_angles in enumerate(optimal_assignments):
    aux_dict = {}
    for j in range(i + 1):
        aux_dict[f"beta_{j}"] = opt_angles[j]
    for j in range(i + 1):
        aux_dict[f"gamma_{j}"] = opt_angles[j + i + 1]
    
    optimal_assignments_df = pd.DataFrame.from_dict(aux_dict, orient = "index", columns = ["vals"])
    optimal_assignments_df.index.name = "angles"
    optimal_assignments_df.to_csv(f"./results/optimal_angles/train{train_set_length}_val{val_set_length}_1_NelderMead_p{i+1}.csv")

    with open(f"./results/optimal_angles/train{train_set_length}_val{val_set_length}_1_NelderMead_p{i+1}.pkl", "wb") as f:
        pickle.dump(aux_dict, f)


#### COBYLA

In [16]:
val_stats_list = []
optimal_assignments = []
best_alphas = {}
for p in range(1, 11):
    print(f"Para el numero de capas p = {p}")
    
    optimal_params, best_alpha, val_stats = qaoa_pretrain_algorithm(train_set, val_set, epochs= 5, clusterize = False, clusters = 10,
                                                                     delta = 0.75, l = 1, c = 3, min_method = 'COBYLA',
                                                                     p = p, normalizeHc = True,
                                                                    shots = 10_000, seed = 42, verbose = True)
    
    for epoch_idx, epoch_stats in val_stats.items():
        val_stats_list.append([p, epoch_idx] + epoch_stats)
    optimal_assignments.append(optimal_params)
    best_alphas[f"{p}"] = best_alpha
    print(f"Los parametros mas optimos proporcionan un escalado de {best_alpha}")


Para el numero de capas p = 1
Epoch 01 (0174 s): min - 0.422, mean - 0.469, std - 0.023, max - 0.498
Epoch 02 (0174 s): min - 0.444, mean - 0.479, std - 0.026, max - 0.521
Epoch 03 (0174 s): min - 0.425, mean - 0.489, std - 0.045, max - 0.565
Epoch 04 (0172 s): min - 0.416, mean - 0.465, std - 0.034, max - 0.525
Epoch 05 (0167 s): min - 0.424, mean - 0.453, std - 0.018, max - 0.480
Los parametros mas optimos proporcionan un escalado de 0.41582203927144007
Para el numero de capas p = 2
Epoch 01 (0219 s): min - 0.313, mean - 0.443, std - 0.207, max - 1.024
Epoch 02 (0217 s): min - 0.317, mean - 0.402, std - 0.159, max - 0.871
Epoch 03 (0224 s): min - 0.304, mean - 0.355, std - 0.045, max - 0.477
Epoch 04 (0205 s): min - 0.305, mean - 0.333, std - 0.031, max - 0.415
Epoch 05 (0227 s): min - 0.311, mean - 0.391, std - 0.099, max - 0.676
Los parametros mas optimos proporcionan un escalado de 0.30394785628694015
Para el numero de capas p = 3
Epoch 01 (0420 s): min - 0.264, mean - 0.295, std 

In [17]:
val_stats_df = pd.DataFrame(val_stats_list, columns = ['p', 'epoch', 'min_alpha', 'mean_alpha', 'std_alpha', 'max_alpha', 'runtime'])
val_stats_df.to_csv(f"./results/validation_statistics/train{train_set_length}_val{val_set_length}_1_COBYLA.csv", index = False)
best_alphas_df = pd.DataFrame.from_dict(best_alphas, orient = "index", columns=["best_alphas"])
best_alphas_df.index.name = "p"
best_alphas_df.to_csv(f"./results/training_best_alphas/train{train_set_length}_val{val_set_length}_1_COBYLA.csv")

for i, opt_angles in enumerate(optimal_assignments):
    aux_dict = {}
    for j in range(i + 1):
        aux_dict[f"beta_{j}"] = opt_angles[j]
    for j in range(i + 1):
        aux_dict[f"gamma_{j}"] = opt_angles[j + i + 1]
    
    optimal_assignments_df = pd.DataFrame.from_dict(aux_dict, orient = "index", columns = ["vals"])
    optimal_assignments_df.index.name = "angles"
    optimal_assignments_df.to_csv(f"./results/optimal_angles/train{train_set_length}_val{val_set_length}_1_COBYLA_p{i+1}.csv")

    with open(f"./results/optimal_angles/train{train_set_length}_val{val_set_length}_1_COBYLA_p{i+1}.pkl", "wb") as f:
        pickle.dump(aux_dict, f)
